# Proyecto 2 - Sistema de Ciberseguridad con IA

## Task 1: Configuración Segura de la Red (CSP)

### Task 1.1 Modelado

El problema se modela como un CSP (Constraint Satisfaction Problem):

**Variables:** Cada nodo del grafo representa un servidor. Hay entre 15 y 20 nodos. Cada variable $X_i$ corresponde al servidor $i$.

**Dominios:** Cada variable puede tomar uno de 4 protocolos de seguridad: `{Rojo, Verde, Azul, Amarillo}`.

**Restricciones (Factores):** Para cada arista $(X_i, X_j)$ en el grafo, el factor $f_{ij}$ exige que $X_i \neq X_j$. Esto evita vulnerabilidades de movimiento lateral entre servidores directamente conectados.

En el modelo de **Factor Graph**, los nodos de variable son los servidores y los nodos de factor representan cada restricción de adyacencia.

In [144]:
import random
import time
import copy

random.seed(42)

# Numero de nodos entre 15 y 20
NUM_NODOS = random.randint(15, 20)
COLORES = ["Rojo", "Verde", "Azul", "Amarillo"]

# Generar grafo aleatorio como lista de adyacencia
def generar_grafo(n, prob_arista=0.3):
    grafo = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            if random.random() < prob_arista:
                grafo[i].add(j)
                grafo[j].add(i)
    # Asegurar que el grafo sea conexo
    for i in range(1, n):
        if not grafo[i]:
            j = random.randint(0, i - 1)
            grafo[i].add(j)
            grafo[j].add(i)
    return grafo

GRAFO = generar_grafo(NUM_NODOS)

print(f"Grafo generado con {NUM_NODOS} nodos")
print(f"Dominio de colores: {COLORES}")
print("Lista de adyacencia:")
for nodo, vecinos in GRAFO.items():
    print(f"  Nodo {nodo}: {sorted(vecinos)}")

Grafo generado con 20 nodos
Dominio de colores: ['Rojo', 'Verde', 'Azul', 'Amarillo']
Lista de adyacencia:
  Nodo 0: [1, 3, 4, 5, 9, 10, 11, 18]
  Nodo 1: [0, 3, 5, 6, 8, 12, 13, 15, 17, 18]
  Nodo 2: [5, 6, 15]
  Nodo 3: [0, 1, 4, 7, 9, 12, 15, 19]
  Nodo 4: [0, 3, 8, 12, 13]
  Nodo 5: [0, 1, 2, 7, 11, 12, 18]
  Nodo 6: [1, 2, 9, 14, 15, 16]
  Nodo 7: [3, 5, 10, 14, 16]
  Nodo 8: [1, 4, 11, 15, 16, 18]
  Nodo 9: [0, 3, 6, 13, 16]
  Nodo 10: [0, 7]
  Nodo 11: [0, 5, 8, 12, 13, 14, 17, 19]
  Nodo 12: [1, 3, 4, 5, 11, 13, 14, 17]
  Nodo 13: [1, 4, 9, 11, 12, 17]
  Nodo 14: [6, 7, 11, 12, 15]
  Nodo 15: [1, 2, 3, 6, 8, 14, 16, 19]
  Nodo 16: [6, 7, 8, 9, 15, 19]
  Nodo 17: [1, 11, 12, 13, 18]
  Nodo 18: [0, 1, 5, 8, 17, 19]
  Nodo 19: [3, 11, 15, 16, 18]


### Task 1.2 Backtracking Puro

El **Backtracking Search** asigna valores a las variables una por una. Si en algún punto la asignación viola una restriccion, retrocede e intenta otro valor. No realiza ninguna inferencia anticipada.

In [145]:
def es_consistente(nodo, color, asignacion, grafo):
    # Verifica que ningun vecino ya asignado tenga el mismo color
    for vecino in grafo[nodo]:
        if vecino in asignacion and asignacion[vecino] == color:
            return False
    return True

def backtracking_puro(asignacion, nodos, grafo, colores, contador):
    # Caso base: todos los nodos asignados
    if len(asignacion) == len(nodos):
        return asignacion

    # Seleccionar el primer nodo no asignado
    nodo = next(n for n in nodos if n not in asignacion)

    for color in colores:
        contador[0] += 1
        if es_consistente(nodo, color, asignacion, grafo):
            asignacion[nodo] = color
            resultado = backtracking_puro(asignacion, nodos, grafo, colores, contador)
            if resultado is not None:
                return resultado
            del asignacion[nodo]
    return None

nodos = list(GRAFO.keys())
contador_puro = [0]
t0 = time.time()
solucion_pura = backtracking_puro({}, nodos, GRAFO, COLORES, contador_puro)
t1 = time.time()
tiempo_puro = t1 - t0

print("Backtracking puro - Solucion encontrada:")
for nodo, color in sorted(solucion_pura.items()):
    print(f"  Servidor {nodo}: {color}")
print(f"Asignaciones intentadas: {contador_puro[0]}")
print(f"Tiempo: {tiempo_puro:.6f} segundos")

Backtracking puro - Solucion encontrada:
  Servidor 0: Rojo
  Servidor 1: Verde
  Servidor 2: Rojo
  Servidor 3: Azul
  Servidor 4: Verde
  Servidor 5: Azul
  Servidor 6: Azul
  Servidor 7: Amarillo
  Servidor 8: Rojo
  Servidor 9: Amarillo
  Servidor 10: Verde
  Servidor 11: Verde
  Servidor 12: Amarillo
  Servidor 13: Rojo
  Servidor 14: Rojo
  Servidor 15: Amarillo
  Servidor 16: Verde
  Servidor 17: Azul
  Servidor 18: Amarillo
  Servidor 19: Rojo
Asignaciones intentadas: 1972
Tiempo: 0.001062 segundos


### Task 1.3 Forward Checking

El **Forward Checking** mejora el backtracking al propagar restricciones hacia adelante. Cuando se asigna un valor a una variable, se eliminan los valores incompatibles del dominio de sus vecinos no asignados. Si algún dominio queda vacío, se retrocede inmediatamente sin continuar.

In [146]:
def forward_checking(nodo, color, dominios, grafo):
    # Elimina el color asignado del dominio de los vecinos no asignados
    eliminados = []
    for vecino in grafo[nodo]:
        if color in dominios[vecino]:
            dominios[vecino].remove(color)
            eliminados.append(vecino)
            # Dominio vacio implica fallo
            if len(dominios[vecino]) == 0:
                return False, eliminados
    return True, eliminados

def restaurar_dominios(nodo_color, eliminados, dominios):
    # Restaura los dominios al deshacer una asignacion
    for vecino in eliminados:
        dominios[vecino].add(nodo_color)

print("Forward Checking implementado correctamente.")
print("Propaga restricciones al asignar un valor, eliminando valores incompatibles en vecinos.")

Forward Checking implementado correctamente.
Propaga restricciones al asignar un valor, eliminando valores incompatibles en vecinos.


### Task 1.4 Heurística MCV (Minimum Remaining Values)

La heurística **MCV** (o Variable más Restringida) selecciona siempre la variable con el menor numero de valores validos restantes en su dominio. Esto reduce el factor de ramificación del arbol de busqueda al tratar primero las variables con menos opciones, detectando fallos mas temprano.

In [147]:
def seleccionar_variable_mcv(asignacion, dominios):
    # Elige el nodo no asignado con el dominio mas pequeno (MCV)
    no_asignados = [n for n in dominios if n not in asignacion]
    return min(no_asignados, key=lambda n: len(dominios[n]))

print("Heuristica MCV implementada.")
print("Selecciona la variable con menor cantidad de valores validos restantes.")

Heuristica MCV implementada.
Selecciona la variable con menor cantidad de valores validos restantes.


### Task 1.5 Backtracking Optimizado (Forward Checking + MCV)

Se combinan Forward Checking y MCV para crear un backtracking eficiente: MCV escoge la variable mas critica y Forward Checking poda los dominios anticipadamente, evitando explorar ramas condenadas al fracaso.

In [148]:
def backtracking_optimizado(asignacion, dominios, grafo, colores, contador):
    # Todos los nodos asignados
    if len(asignacion) == len(grafo):
        return asignacion

    # Seleccion por MCV
    nodo = seleccionar_variable_mcv(asignacion, dominios)

    for color in list(dominios[nodo]):
        contador[0] += 1
        if es_consistente(nodo, color, asignacion, grafo):
            asignacion[nodo] = color

            # Forward checking: actualizar dominios de vecinos
            ok, eliminados = forward_checking(nodo, color, dominios, grafo)

            if ok:
                resultado = backtracking_optimizado(asignacion, dominios, grafo, colores, contador)
                if resultado is not None:
                    return resultado

            # Deshacer asignacion y restaurar dominios
            del asignacion[nodo]
            restaurar_dominios(color, eliminados, dominios)

    return None

# Inicializar dominios completos para cada nodo
dominios_iniciales = {nodo: set(COLORES) for nodo in GRAFO}
contador_opt = [0]
t2 = time.time()
solucion_opt = backtracking_optimizado({}, dominios_iniciales, GRAFO, COLORES, contador_opt)
t3 = time.time()
tiempo_opt = t3 - t2

print("Backtracking optimizado (FC + MCV) - Solucion encontrada:")
for nodo, color in sorted(solucion_opt.items()):
    print(f"  Servidor {nodo}: {color}")
print(f"Asignaciones intentadas: {contador_opt[0]}")
print(f"Tiempo: {tiempo_opt:.6f} segundos")

Backtracking optimizado (FC + MCV) - Solucion encontrada:
  Servidor 0: Rojo
  Servidor 1: Azul
  Servidor 2: Rojo
  Servidor 3: Amarillo
  Servidor 4: Azul
  Servidor 5: Amarillo
  Servidor 6: Amarillo
  Servidor 7: Verde
  Servidor 8: Rojo
  Servidor 9: Verde
  Servidor 10: Azul
  Servidor 11: Azul
  Servidor 12: Verde
  Servidor 13: Rojo
  Servidor 14: Rojo
  Servidor 15: Verde
  Servidor 16: Azul
  Servidor 17: Amarillo
  Servidor 18: Verde
  Servidor 19: Rojo
Asignaciones intentadas: 25
Tiempo: 0.000207 segundos
